# VK Dataset Shrink

Урезает уже готовый `data/VK/inter.json` под более быстрый эксперимент. Применяется поверх артефактов Vk-пайплайна (см. `notebooks/VkDatasetProcessing.ipynb`), без перекачки HF-датасета и без пересчёта эмбедов.

Два рычага:
- `MAX_HISTORY_PER_USER = K` — оставить только последние K интеракций на юзера.
- `USER_SUBSAMPLE_RATIO = R` — случайный сабсэмпл доли юзеров (1.0 = все).

После truncation re-Core-5 + dense remap, чтобы id-ы были плотными и согласованными между `inter.json` и `content_embeddings.pkl`.

Подробное обоснование цифр — в `ai/vk_exps/vk_plan4_smaller_dataset.md`.

In [ ]:
import os
import json
import pickle
import random
import numpy as np

## Параметры

In [ ]:
INPUT_DIR  = '../data/VK'
OUTPUT_DIR = '../data/VK_small'

MAX_HISTORY_PER_USER = 15      # K
USER_SUBSAMPLE_RATIO = 1.0     # R; 1.0 = все юзеры
RANDOM_SEED = 42
KEEP_LAST = True               # True = items[-K:], всегда True для рекомендалки

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'INPUT_DIR={INPUT_DIR}, OUTPUT_DIR={OUTPUT_DIR}')
print(f'K={MAX_HISTORY_PER_USER}, R={USER_SUBSAMPLE_RATIO}, seed={RANDOM_SEED}')

## Загрузка inter.json

In [ ]:
with open(os.path.join(INPUT_DIR, 'inter.json'), 'r') as f:
    user_interactions = json.load(f)

orig_users = len(user_interactions)
orig_items = max(max(v) for v in user_interactions.values()) + 1
orig_inter = sum(len(v) for v in user_interactions.values())
print(f'before: users={orig_users:,}, num_items={orig_items:,}, total_inter={orig_inter:,}')

## Сабсэмпл юзеров и truncation

`random.seed(RANDOM_SEED)` фиксирует выбор юзеров — все downstream-стадии (cf_dataset_builder, cf_finetune, ...) увидят один и тот же `inter.json`.

In [ ]:
random.seed(RANDOM_SEED)
user_ids = list(user_interactions.keys())
if USER_SUBSAMPLE_RATIO < 1.0:
    keep_n = int(len(user_ids) * USER_SUBSAMPLE_RATIO)
    user_ids = random.sample(user_ids, keep_n)

shrunk = {}
for uid in user_ids:
    seq = user_interactions[uid]
    seq = seq[-MAX_HISTORY_PER_USER:] if KEEP_LAST else seq[:MAX_HISTORY_PER_USER]
    shrunk[uid] = seq

print(f'after subsample+truncate: users={len(shrunk):,}, total_inter={sum(len(v) for v in shrunk.values()):,}')

## Re-Core-5 + dense remap

После truncation некоторые айтемы могут потерять интеракции (жили только в обрезанных «головах»), и появятся юзеры с <5 элементов. Итеративный Core-5 до сходимости — точно как в `notebooks/DatasetProcessing.ipynb`. Затем плотный remap `item_id -> 0..M-1`, `user_id -> 0..N-1`.

In [ ]:
def core5_filter(d):
    while True:
        item_counts = {}
        for seq in d.values():
            for it in seq:
                item_counts[it] = item_counts.get(it, 0) + 1
        bad_items = {it for it, c in item_counts.items() if c < 5}
        new_d = {}
        for u, seq in d.items():
            cleaned = [it for it in seq if it not in bad_items]
            if len(cleaned) >= 5:
                new_d[u] = cleaned
        if new_d == d:
            return new_d
        d = new_d

shrunk = core5_filter(shrunk)
print(f'after Core-5: users={len(shrunk):,}, total_inter={sum(len(v) for v in shrunk.values()):,}')

In [ ]:
old_items = sorted({it for seq in shrunk.values() for it in seq})
item_remap = {old: new for new, old in enumerate(old_items)}

old_users = sorted(shrunk.keys(), key=lambda x: int(x))
user_remap = {old: new for new, old in enumerate(old_users)}

remapped = {
    str(user_remap[u]): [item_remap[it] for it in seq]
    for u, seq in shrunk.items()
}

new_users = len(remapped)
new_items = len(old_items)
new_inter = sum(len(v) for v in remapped.values())
print(f'after remap: users={new_users:,}, num_items={new_items:,}, total_inter={new_inter:,}')

# инварианты для downstream
assert min(len(v) for v in remapped.values()) >= 5, 'Core-5 broken'
assert set(int(k) for k in remapped.keys()) == set(range(new_users)), 'user_ids not dense'
assert set(it for v in remapped.values() for it in v) == set(range(new_items)), 'item_ids not dense'

## Сохранение inter.json

In [ ]:
inter_out = os.path.join(OUTPUT_DIR, 'inter.json')
with open(inter_out, 'w') as f:
    json.dump(remapped, f)
print(f'saved: {inter_out}')

## Фильтрация и сохранение content_embeddings.pkl

Старый pkl содержит эмбеды на исходный набор `item_id`. Берём только выжившие после Core-5 айтемы и кладём в новый порядок `0..new_items-1`.

In [ ]:
with open(os.path.join(INPUT_DIR, 'content_embeddings.pkl'), 'rb') as f:
    emb = pickle.load(f)

emb_by_old = dict(zip(emb['item_id'], emb['embedding']))

new_emb = {'item_id': list(range(new_items)), 'embedding': [None] * new_items}
for old_id, new_id in item_remap.items():
    new_emb['embedding'][new_id] = emb_by_old[old_id]
assert all(e is not None for e in new_emb['embedding'])

emb_out = os.path.join(OUTPUT_DIR, 'content_embeddings.pkl')
with open(emb_out, 'wb') as f:
    pickle.dump(new_emb, f, protocol=pickle.HIGHEST_PROTOCOL)

dim = np.asarray(new_emb['embedding'][0]).shape[-1]
print(f'saved: {emb_out} (num_items={new_items}, dim={dim})')

## Прогноз train-сэмплов

Для `is_extended=True` (см. `tiger/modeling/dataset/base.py`) кол-во train-сэмплов = `Σ max(0, len-3)`. Сравниваем с исходными 1 135 267.

In [ ]:
extended_samples = sum(max(0, len(v) - 3) for v in remapped.values())
print(f'TIGER train samples (is_extended=True): {extended_samples:,}')
print(f'reduction vs original: {extended_samples / 1_135_267 * 100:.1f}%')

# распределение длин — sanity на cold/warm
import collections
hist = collections.Counter(len(v) for v in remapped.values())
print('length histogram (top):')
for L in sorted(hist)[:10] + sorted(hist)[-3:]:
    print(f'  len={L}: {hist[L]:,}')